In [33]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [25]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [5]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [39]:
cnn = Website("https://www.cnn.com/")
cnn.links

['https://www.cnn.com',
 'https://www.cnn.com/us',
 'https://www.cnn.com/world',
 'https://www.cnn.com/politics',
 'https://www.cnn.com/business',
 'https://www.cnn.com/health',
 'https://www.cnn.com/entertainment',
 'https://www.cnn.com/style',
 'https://www.cnn.com/travel',
 'https://www.cnn.com/sports',
 'https://www.cnn.com/science',
 'https://www.cnn.com/climate',
 'https://www.cnn.com/weather',
 'https://www.cnn.com/world/europe/ukraine',
 'https://www.cnn.com/world/middleeast/israel',
 'https://www.cnn.com/cnn-underscored',
 'https://www.cnn.com/games',
 'https://www.cnn.com/us',
 'https://www.cnn.com/world',
 'https://www.cnn.com/politics',
 'https://www.cnn.com/business',
 'https://www.cnn.com/health',
 'https://www.cnn.com/entertainment',
 'https://www.cnn.com/style',
 'https://www.cnn.com/travel',
 'https://www.cnn.com/sports',
 'https://www.cnn.com/science',
 'https://www.cnn.com/climate',
 'https://www.cnn.com/weather',
 'https://www.cnn.com/world/europe/ukraine',
 'https:

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [60]:
link_system_prompt = "You are provided with a list of links found on a News Webpage and you need to categorize the links based on certain topics. \
You are able to decide which of the links would be most relevant to the topics provided to you and select the best link for each category provided to you. \
Each category can have only a single link\
The URLs should only start with http or https, should be a valid URL\
The categories could be like USA, World, Politics, Entertainment, Sports, Finance, etc.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "politics", "url": "https://politics.url/goes/here/about"},
        {"type": "sports": "url": "https://sports.url/goes/here/about"}
    ]
}
"""

In [61]:
print(link_system_prompt)

You are provided with a list of links found on a News Webpage and you need to categorize the links based on certain topics. You are able to decide which of the links would be most relevant to the topics provided to you and select the best link for each category provided to you. Each category can have only a single linkThe URLs should only start with http or https, should be a valid URLThe categories could be like USA, World, Politics, Entertainment, Sports, Finance, etc.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "politics", "url": "https://politics.url/goes/here/about"},
        {"type": "sports": "url": "https://sports.url/goes/here/about"}
    ]
}



In [62]:
def get_links_user_prompt(website, categories):
    user_prompt = f"Here is the list of links on the website of - {website.url}"
    user_prompt += f"Here is the list of categories provided - {categories}"
    user_prompt += "please categorize the links based on the categories provided, respond with the full https URL in JSON format. \
    The URLs should only start with http or https, should be a valid URL\
Do not include Terms of Service, Privacy, email links, non-https links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [63]:
print(get_links_user_prompt(cnn, ["sports", "finance"]))

Here is the list of links on the website of - https://www.cnn.com/Here is the list of categories provided - ['sports', 'finance']please categorize the links based on the categories provided, respond with the full https URL in JSON format.     The URLs should only start with http or https, should be a valid URLDo not include Terms of Service, Privacy, email links, non-https links.
Links (some might be relative links):
https://www.cnn.com
https://www.cnn.com/us
https://www.cnn.com/world
https://www.cnn.com/politics
https://www.cnn.com/business
https://www.cnn.com/health
https://www.cnn.com/entertainment
https://www.cnn.com/style
https://www.cnn.com/travel
https://www.cnn.com/sports
https://www.cnn.com/science
https://www.cnn.com/climate
https://www.cnn.com/weather
https://www.cnn.com/world/europe/ukraine
https://www.cnn.com/world/middleeast/israel
https://www.cnn.com/cnn-underscored
https://www.cnn.com/games
https://www.cnn.com/us
https://www.cnn.com/world
https://www.cnn.com/politics
ht

In [64]:
def get_links(url, categories):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website, categories)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [65]:
get_links("https://www.cnn.com", ["sports", "finance"])

{'links': [{'type': 'sports', 'url': 'https://www.cnn.com/sports'},
  {'type': 'finance', 'url': 'https://www.cnn.com/business'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [66]:
def get_all_details(url, categories):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url, categories)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [67]:
print(get_all_details("https://www.cnn.com", ["sports", "finance"]))

Found links: {'links': [{'type': 'sports', 'url': 'https://www.cnn.com/sports'}, {'type': 'finance', 'url': 'https://www.cnn.com/business'}]}
Landing page:
Webpage Title:
Breaking News, Latest News and Videos | CNN
Webpage Contents:
CNN values your feedback
1. How relevant is this ad to you?
2. Did you encounter any technical issues?
Video player was slow to load content
Video content never loaded
Ad froze or did not finish loading
Video content did not start after ad
Audio on ad was too loud
Other issues
Ad never loaded
Ad prevented/slowed the page from loading
Content moved around while ad loaded
Ad was repetitive to ads I've seen previously
Other issues
Cancel
Submit
Thank You!
Your effort and contribution in providing this feedback is much
                                        appreciated.
Close
Ad Feedback
Close icon
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
Ukraine-Russia War
Israel-Hamas War
Underscored
Games
More
US
World
Poli

In [74]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a news website \
Try to cover the most latest news \
and creates a short summary for each news category in about 200 words, try to cover the most hot and latest news for each category provided.\
Respond in markdown.\
You will be provided with the content of the news website, the categories for which you need to summarize the news for"

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [75]:
def get_news_user_prompt(news_website, url, categories):
    user_prompt = f"You are looking at a news website called: {news_website}\n"
    user_prompt += f"Here are the categories based on which you need to categorize the news content on: {categories}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short summary of the news website based on categories in markdown. Try to cover the most latest news \\n"
    user_prompt += get_all_details(url, categories)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [76]:
get_news_user_prompt("CNN", "https://cnn.com", ["sports", "finance"])

Found links: {'links': [{'type': 'sports', 'url': 'https://www.cnn.com/sports'}, {'type': 'finance', 'url': 'https://www.cnn.com/business'}]}


"You are looking at a news website called: CNN\nHere are the categories based on which you need to categorize the news content on: ['sports', 'finance']\nHere are the contents of its landing page and other relevant pages; use this information to build a short summary of the news website based on categories in markdown. Try to cover the most latest news \\nLanding page:\nWebpage Title:\nBreaking News, Latest News and Videos | CNN\nWebpage Contents:\nCNN values your feedback\n1. How relevant is this ad to you?\n2. Did you encounter any technical issues?\nVideo player was slow to load content\nVideo content never loaded\nAd froze or did not finish loading\nVideo content did not start after ad\nAudio on ad was too loud\nOther issues\nAd never loaded\nAd prevented/slowed the page from loading\nContent moved around while ad loaded\nAd was repetitive to ads I've seen previously\nOther issues\nCancel\nSubmit\nThank You!\nYour effort and contribution in providing this feedback is much\n        

In [77]:
def create_summary(company_name, url, categories):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_news_user_prompt(company_name, url, categories)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [78]:
create_summary("CNN", "https://cnn.com", ["sports", "finance"])

Found links: {'links': [{'type': 'sports', 'url': 'https://www.cnn.com/sports'}, {'type': 'finance', 'url': 'https://www.cnn.com/business'}]}


# CNN News Summary

## Sports
The sports section is currently highlighting significant developments in various leagues. As the NFL season progresses, teams are jostling for playoff positions, and analysts are providing insights into key matchups this weekend. Additionally, college football is gearing up for the final stretch before the bowl games, with critical rankings updating after last week's games. In basketball, the NBA is witnessing impressive performances from rookies while established stars like LeBron James and Stephen Curry are setting new records. There's also ongoing excitement for the upcoming Winter Olympics, with Team USA unveiling its roster in preparation for the international competition.

## Finance
In finance news, markets are experiencing volatility amidst ongoing concerns regarding inflation and interest rate hikes. Analysts are watching the Federal Reserve's moves closely as indicators suggest potential shifts in economic policy. The recent earnings reports from major tech companies have shown mixed results, prompting discussions on the tech sector's direction. Moreover, while the Biden administration is pushing for policies that address economic inequality, the stock market's response remains uncertain. Investors are advised to remain cautious as geopolitical tensions, especially related to the Israel-Hamas conflict, might also impact global markets and economic stability.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'home page', 'url': 'https://huggingface.co/'}, {'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community discussion page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


# Hugging Face Company Brochure

---

## About Us
**Hugging Face** is at the forefront of the AI revolution, fostering a vibrant community dedicated to building and advancing machine learning technologies. Our platform is the ultimate hub for collaboration, allowing users to create, discover, and innovate with a vast repository of models, datasets, and applications.

### Our Mission:
*To empower the AI community in building the future by providing tools and resources for collaborative machine learning.*

---

## What We Offer
### Extensive Resources
- **Models**: More than 1 million AI models ready for exploration and implementation.
- **Datasets**: Access over 250,000 datasets for various ML tasks, enhancing research and development.
- **Spaces**: Collaborate and explore innovative applications within a unified environment.

### Enterprise Solutions
We cater to businesses of all sizes with enterprise-grade tools ensuring security and efficiency:
- Advanced Compute options
- Optimized Inference Endpoints
- Flexible pricing starting from **$20/user/month**

### Open Source Tools
Join us in creating the future of ML with our comprehensive open-source stack:
- **Transformers**: Cutting-edge models for Pytorch, TensorFlow, and JAX.
- **Diffusers**: State-of-the-art diffusion models for image and audio.
- **PEFT**: Efficient methods for fine-tuning large models.

---

## Our Customers
With a growing community of over **50,000 organizations**, Hugging Face is trusted by industry leaders, including:
- **Meta**
- **Amazon Web Services**
- **Google**
- **Microsoft**
- **Grammarly**

These collaborations make ours the backbone for many AI applications across various industries.

---

## Company Culture
At Hugging Face, we value **collaboration, inclusivity, and innovation**. Our culture promotes:
- A learning environment where team members can expand their expertise.
- Open communication and community-driven projects.
- A focus on ethical AI practices ensuring responsible development.

---

## Careers at Hugging Face
Join our team and contribute to a mission-driven organization where your work can have a significant impact on the future of AI. We are looking for passionate individuals across various fields:
- Software Engineers
- Data Scientists
- ML Researchers
- Product Managers

Explore available career opportunities on our [Jobs Page](#).

---

### Get Involved!
Whether you're a prospective customer, investor, or recruit, Hugging Face welcomes you to be part of our journey in shaping the future of artificial intelligence.

**Connect with us on**
- [GitHub](https://github.com/huggingface)
- [Twitter](https://twitter.com/huggingface)
- [LinkedIn](https://www.linkedin.com/company/huggingface)
- [Discord](https://discord.com/invite/huggingface)

Let's build the future of AI together! 

--- 

For more information, visit our website: [huggingface.co](https://huggingface.co)

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 2 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>